In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import random
import numpy as np
import pandas as pd

In [2]:
# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
# OUTPUT_DIR = Path(__file__).resolve().parent

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def simpson_wcc(n_cycles: int) -> np.ndarray:
    weights = []
    for i in range(n_cycles + 1):
        if i == 0 or i == n_cycles:
            weights.append(1.0 / 3.0)
        elif i % 2 == 1:
            weights.append(4.0 / 3.0)
        else:
            weights.append(2.0 / 3.0)
    return np.array(weights, dtype=float)


def prob_to_rate(p: float) -> float:
    p = float(np.clip(p, 1e-12, 1.0 - 1e-12))
    return -np.log(1.0 - p)


def rates_to_row_probs(rates: dict[str, float], dt: float) -> dict[str, float]:
    total_rate = float(sum(rates.values()))
    if total_rate <= 0:
        out = {k: 0.0 for k in rates}
        out["stay"] = 1.0
        return out
    move_prob = 1.0 - np.exp(-total_rate * dt)
    row = {k: (rates[k] / total_rate) * move_prob for k in rates}
    row["stay"] = 1.0 - move_prob
    return row


def clamp01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def safe_icer(cost: float, effect: float) -> float:
    return np.nan if effect <= 0 else float(cost) / float(effect)


In [3]:
# ---------------------------------------------------------------------
# 1) Cost Inputs reported by PSI (USD, constant 2024)
# ---------------------------------------------------------------------

# Year 1 Cost inputs (2024 USD)
program_manager = 491.34 * 12
logistics_super = 640.85 * 12
mne_officer = 160.00 * 12
staff_support = 932.91 * 12
other_direct = 330.85 * 12

startup_field_travel = 7_862.66
legacy_startup_2022 = 20_000.00
comm_materials = 2_500.00

training_initial = 8_000.00
training_refresh = 4_000.00

drone1_rental = 115_200.00
drone2_rental = 80_000.00
takeoff_point = 60_000.00
packing_material = 13_200.00

minsan_local = 5_000.00
supervision_visits = 28_966.67

km_avoided_y1 = 45_000
cost_per_km = 1.26
wastage_saved_y1 = 12_500
labour_hours_saved = 3_200
hourly_wage = 1.50

offsets_y1 = (
    km_avoided_y1 * cost_per_km
    + wastage_saved_y1
    + labour_hours_saved * hourly_wage
)

base_program_admin_y1 = program_manager + logistics_super + staff_support + other_direct
base_monitoring_y1 = mne_officer + minsan_local + supervision_visits
base_startup_y1 = startup_field_travel + legacy_startup_2022 + comm_materials
base_training_y1 = training_initial + training_refresh

observed_drone_month_fee = (drone1_rental + drone2_rental) / (12 + 10)
observed_top_month_fee = takeoff_point / 12.0
packing_cost_per_delivery = packing_material / 1760.0


In [4]:
# ---------------------------------------------------------------------
# 2) Ten-year scale-up schedule and cost scenarios
# ---------------------------------------------------------------------

# Scale up Scenario Costs reported by PSI

mixed_drone_month_fee = 7_500.00
mixed_overage_fee = 110.00
mixed_threshold_flights = 75.0

at_scale_drone_month_fee = 4_800.00
at_scale_top_month_fee = 3_600.00

YEARS = 10
DISCOUNT_RATE = 0.03
cycle_length = 1.0 / 12.0
n_cycles = YEARS * 12
H, U, S, D = 0, 1, 2, 3

COMMON_DRONES_BY_YEAR = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20], dtype=float)
COMMON_TOPS_BY_YEAR = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)

SCENARIOS = [
    {
        "Scenario_ID": 1,
        "Scenario": "Expensive pilot pricing + underused network",
        "Price_regime": "Observed pilot/FID pricing",
        "Support_model": "Pilot-style support",
        "Description": "Observed pilot pricing is retained and the network is underused.",
        "drone_month_fee": observed_drone_month_fee,
        "top_month_fee": observed_top_month_fee,
        "overage_fee": 0.0,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 55.0,
        "support_multiplier_year2plus": 1.00,
    },
    {
        "Scenario_ID": 2,
        "Scenario": "Transitional GAVI pricing + expected utilisation",
        "Price_regime": "Transitional mixed/GAVI pricing",
        "Support_model": "Transition to routine delivery",
        "Description": "GAVI-style mixed contract with expected drone use; deterministic base case.",
        "drone_month_fee": mixed_drone_month_fee,
        "top_month_fee": observed_top_month_fee,
        "overage_fee": mixed_overage_fee,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 80.0,
        "support_multiplier_year2plus": 0.85,
    },
    {
        "Scenario_ID": 3,
        "Scenario": "At-scale pricing + efficient scale-up",
        "Price_regime": "Negotiated at-scale pricing",
        "Support_model": "Routine implementation",
        "Description": "Lower drone and TOP prices with high utilisation at scale.",
        "drone_month_fee": at_scale_drone_month_fee,
        "top_month_fee": at_scale_top_month_fee,
        "overage_fee": 0.0,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 90.0,
        "support_multiplier_year2plus": 0.70,
    },
]


In [5]:
# ---------------------------------------------------------------------
# 3) trial, Natural History, and Effectiveness inputs
# ---------------------------------------------------------------------

facilities_drone = 54
catchment_per_facility = 9_000
u5_share = 0.18
initial_u5_population = facilities_drone * catchment_per_facility * u5_share

total_population = facilities_drone * catchment_per_facility
crude_birth_rate_per_1000 = 30.0
births_per_month = (crude_birth_rate_per_1000 / 1000.0) * total_population / 12.0

BASE_PARAMS = {
    "p_HD": 0.007,
    "p_HU": 0.40,
    "p_UH": 0.861,
    "p_US": 0.13,
    "p_UD": 0.009,
    "p_SH": 0.938,
    "p_SD": 0.062,
    "u_U": 0.006,
    "u_S": 0.133,
    "LE0": 62.0,
    "control_rdt_in_stock": 0.81,
    "delta_rdt_in_stock": 0.15,
    "control_test_if_stock": 0.60,
    "delta_test_if_stock": 0.19,
    "control_act_if_positive": 0.82,
    "pass_through": 0.82,
    "medical_cost_scale": 1.0,
    "act_eff": 0.96,
}

MONTHLY_DISCOUNT_10Y = 1.0 / np.power(1.0 + DISCOUNT_RATE * cycle_length, np.arange(n_cycles + 1))
MONTHLY_DISCOUNT_1Y = 1.0 / np.power(1.0 + DISCOUNT_RATE * cycle_length, np.arange(13))
CYCLE_WEIGHTS_10Y = simpson_wcc(n_cycles)
CYCLE_WEIGHTS_1Y = simpson_wcc(12)
MEDICAL_COST_BASE = 16.25
OTHER_HEALTH_COSTS = 30.29

In [6]:
# ---------------------------------------------------------------------
# 4) Build 10 year Cost Schedules
# ---------------------------------------------------------------------

def build_cost_schedule(scenario: dict) -> pd.DataFrame:
    rows = []
    prev_tops = 0.0

    for year in range(1, YEARS + 1):
        drones = COMMON_DRONES_BY_YEAR[year - 1]
        tops = COMMON_TOPS_BY_YEAR[year - 1]
        new_tops = tops - prev_tops

        if year == 1:
            flights_per_drone_month = 80.0
            annual_deliveries = 1760.0
            delivery_scale = 1.0
            utilisation_scale = 1.0

            program_admin = base_program_admin_y1
            targeting = base_startup_y1
            training = base_training_y1
            monitoring = base_monitoring_y1
            implementation = drone1_rental + drone2_rental + takeoff_point + packing_material
        else:
            flights_per_drone_month = scenario["flights_per_drone_month_year2plus"]
            annual_deliveries = drones * 12.0 * flights_per_drone_month
            delivery_scale = annual_deliveries / 1760.0
            utilisation_scale = min(1.0, flights_per_drone_month / 80.0)
            support_multiplier = scenario["support_multiplier_year2plus"]

            program_admin = base_program_admin_y1 * tops * support_multiplier
            targeting = (startup_field_travel + comm_materials) * new_tops * support_multiplier
            training = training_initial * new_tops * support_multiplier
            if year in [3, 6, 9]:
                training += training_refresh * tops * support_multiplier
            monitoring = base_monitoring_y1 * tops * support_multiplier

            overage_flights_per_drone_month = max(0.0, flights_per_drone_month - scenario["overage_threshold"])
            implementation = (
                scenario["drone_month_fee"] * 12.0 * drones
                + scenario["top_month_fee"] * 12.0 * tops
                + packing_cost_per_delivery * annual_deliveries
                + scenario["overage_fee"] * overage_flights_per_drone_month * 12.0 * drones
            )

        direct_cost = program_admin + targeting + training + implementation + monitoring
        overhead = 0.10 * direct_cost
        gross_cost = direct_cost + overhead
        offsets = offsets_y1 * delivery_scale
        net_cost = gross_cost - offsets
        net_cost_pv = net_cost / ((1.0 + DISCOUNT_RATE) ** (year - 1))

        rows.append({
            "Year": year,
            "Drones": drones,
            "TOPs": tops,
            "Flights_per_drone_month": flights_per_drone_month,
            "Annual_deliveries": annual_deliveries,
            "Delivery_scale_vs_year1": delivery_scale,
            "Utilisation_scale": utilisation_scale,
            "Program_admin": program_admin,
            "Targeting": targeting,
            "Training": training,
            "Implementation": implementation,
            "Monitoring": monitoring,
            "Overhead": overhead,
            "Gross_cost": gross_cost,
            "Offsets": offsets,
            "Net_cost": net_cost,
            "Net_cost_PV": net_cost_pv,
        })
        prev_tops = tops

    df = pd.DataFrame(rows)
    df["Cumulative_net_cost_PV"] = df["Net_cost_PV"].cumsum()
    return df

In [7]:
# ---------------------------------------------------------------------
# 5) Effectiveness model
# ---------------------------------------------------------------------


@dataclass
class TransitionInputs:
    p_HD: float
    p_HU: float
    p_UH: float
    p_US: float
    p_UD: float
    p_SH: float
    p_SD: float
    act_eff: float = 0.96


def service_pathway(
    utilization_scale: float,
    control_rdt_in_stock: float,
    delta_rdt_in_stock: float,
    control_test_if_stock: float,
    delta_test_if_stock: float,
    control_act_if_positive: float,
    pass_through: float,
) -> dict[str, float]:
    stock_control = clamp01(control_rdt_in_stock)
    stock_intervention = clamp01(control_rdt_in_stock + delta_rdt_in_stock * utilization_scale)

    test_control = clamp01(control_test_if_stock)
    test_intervention = clamp01(control_test_if_stock + delta_test_if_stock * utilization_scale)

    p_tested_control = stock_control * test_control
    p_tested_intervention = stock_intervention * test_intervention

    p_prompt_treated_control = p_tested_control * clamp01(control_act_if_positive)
    incremental_testing = max(0.0, p_tested_intervention - p_tested_control)
    p_prompt_treated_intervention = min(
        p_tested_intervention,
        p_prompt_treated_control + incremental_testing * clamp01(pass_through),
    )

    return {
        "P_tested_control": p_tested_control,
        "P_tested_intervention": p_tested_intervention,
        "P_prompt_treated_control": p_prompt_treated_control,
        "P_prompt_treated_intervention": p_prompt_treated_intervention,
    }


def build_transition_matrix(inputs: TransitionInputs, p_prompt_treated: float) -> np.ndarray:
    p_UH_treated = inputs.p_UH + (inputs.p_UD - inputs.p_UD * (1.0 - inputs.act_eff))
    p_US_treated = inputs.p_US
    p_UD_treated = inputs.p_UD * (1.0 - inputs.act_eff)

    P = np.zeros((4, 4), dtype=float)

    healthy_row = rates_to_row_probs(
        {"H->U": prob_to_rate(inputs.p_HU), "H->D": prob_to_rate(inputs.p_HD)},
        cycle_length,
    )
    P[H, H] = healthy_row["stay"]
    P[H, U] = healthy_row["H->U"]
    P[H, D] = healthy_row["H->D"]

    untreated_row = rates_to_row_probs(
        {
            "U->H": prob_to_rate(inputs.p_UH),
            "U->S": prob_to_rate(inputs.p_US),
            "U->D": prob_to_rate(inputs.p_UD),
        },
        cycle_length,
    )
    treated_row = rates_to_row_probs(
        {
            "U->H": prob_to_rate(p_UH_treated),
            "U->S": prob_to_rate(p_US_treated),
            "U->D": prob_to_rate(p_UD_treated),
        },
        cycle_length,
    )

    q = clamp01(p_prompt_treated)
    P[U, H] = (1.0 - q) * untreated_row["U->H"] + q * treated_row["U->H"]
    P[U, U] = (1.0 - q) * untreated_row["stay"] + q * treated_row["stay"]
    P[U, S] = (1.0 - q) * untreated_row["U->S"] + q * treated_row["U->S"]
    P[U, D] = (1.0 - q) * untreated_row["U->D"] + q * treated_row["U->D"]

    severe_row = rates_to_row_probs(
        {"S->H": prob_to_rate(inputs.p_SH), "S->D": prob_to_rate(inputs.p_SD)},
        cycle_length,
    )
    P[S, H] = severe_row["S->H"]
    P[S, S] = severe_row["stay"]
    P[S, D] = severe_row["S->D"]
    P[D, D] = 1.0
    return P


def run_open_cohort(transition_matrices: list[np.ndarray], births_per_month_value: float, initial_u5_value: float) -> np.ndarray:
    n_age_months = 60
    age_weights = np.ones(n_age_months) / n_age_months
    age_state = np.zeros((n_age_months, 4), dtype=float)
    age_state[:, H] = initial_u5_value * age_weights

    trace = np.zeros((len(transition_matrices) + 1, 4), dtype=float)
    trace[0, :] = age_state.sum(axis=0)

    cumulative_deaths_from_aged_out = 0.0
    for t, P in enumerate(transition_matrices):
        age_state = age_state @ P
        aged_out = age_state[-1, :].copy()
        cumulative_deaths_from_aged_out += aged_out[D]

        next_age_state = np.zeros_like(age_state)
        next_age_state[1:, :] = age_state[:-1, :]
        next_age_state[0, H] = births_per_month_value
        age_state = next_age_state

        trace[t + 1, :] = age_state.sum(axis=0)
        trace[t + 1, D] += cumulative_deaths_from_aged_out

    return trace

In [8]:
def run_deterministic_scenario(scenario: dict, pass_through: float | None = None) -> dict:
    params = dict(BASE_PARAMS)
    if pass_through is not None:
        params["pass_through"] = float(pass_through)

    transition_inputs = TransitionInputs(
        p_HD=params["p_HD"],
        p_HU=params["p_HU"],
        p_UH=params["p_UH"],
        p_US=params["p_US"],
        p_UD=params["p_UD"],
        p_SH=params["p_SH"],
        p_SD=params["p_SD"],
        act_eff=params["act_eff"],
    )

    schedule = build_cost_schedule(scenario)
    util_scales = schedule["Utilisation_scale"].to_numpy(dtype=float)
    coverage_scale = np.concatenate(
        ([float(schedule.loc[0, "Delivery_scale_vs_year1"])], np.repeat(schedule["Delivery_scale_vs_year1"].to_numpy(dtype=float), 12))
    )
    program_cost_monthly = np.concatenate(
        ([0.0], np.repeat(schedule["Net_cost"].to_numpy(dtype=float) / 12.0, 12))
    )

    control_path = service_pathway(
        utilization_scale=1.0,
        control_rdt_in_stock=params["control_rdt_in_stock"],
        delta_rdt_in_stock=params["delta_rdt_in_stock"],
        control_test_if_stock=params["control_test_if_stock"],
        delta_test_if_stock=params["delta_test_if_stock"],
        control_act_if_positive=params["control_act_if_positive"],
        pass_through=params["pass_through"],
    )
    P_control = build_transition_matrix(transition_inputs, control_path["P_prompt_treated_control"])
    trace_control = run_open_cohort([P_control] * n_cycles, births_per_month, initial_u5_population)

    intervention_matrices: list[np.ndarray] = []
    for util_scale in util_scales:
        yearly_path = service_pathway(
            utilization_scale=float(util_scale),
            control_rdt_in_stock=params["control_rdt_in_stock"],
            delta_rdt_in_stock=params["delta_rdt_in_stock"],
            control_test_if_stock=params["control_test_if_stock"],
            delta_test_if_stock=params["delta_test_if_stock"],
            control_act_if_positive=params["control_act_if_positive"],
            pass_through=params["pass_through"],
        )
        intervention_matrices += [
            build_transition_matrix(transition_inputs, yearly_path["P_prompt_treated_intervention"])
        ] * 12
    trace_intervention = run_open_cohort(intervention_matrices, births_per_month, initial_u5_population)

    u_vec = np.array([0.0, params["u_U"], params["u_S"], 0.0], dtype=float)
    c_val = (OTHER_HEALTH_COSTS + MEDICAL_COST_BASE) * params["medical_cost_scale"]
    c_val0 = OTHER_HEALTH_COSTS * params["medical_cost_scale"]
    c_vec = np.array([c_val0, c_val, c_val, 0.0], dtype=float)

    yld_control = trace_control.dot(u_vec) * cycle_length
    yld_intervention = trace_intervention.dot(u_vec) * cycle_length

    new_deaths_control = np.diff(trace_control[:, D], prepend=0.0)
    new_deaths_control[new_deaths_control < 0] = 0.0
    new_deaths_intervention = np.diff(trace_intervention[:, D], prepend=0.0)
    new_deaths_intervention[new_deaths_intervention < 0] = 0.0

    pv_le = (1.0 - np.exp(-DISCOUNT_RATE * params["LE0"])) / DISCOUNT_RATE
    yll_control = new_deaths_control * pv_le
    yll_intervention = new_deaths_intervention * pv_le

    dalys_control_cycle = yld_control + yll_control
    dalys_intervention_cycle = yld_intervention + yll_intervention
    medical_cost_control_cycle = trace_control.dot(c_vec) * cycle_length
    medical_cost_intervention_cycle = trace_intervention.dot(c_vec) * cycle_length

    incremental_dalys_cycle = (dalys_control_cycle - dalys_intervention_cycle) * coverage_scale
    incremental_medical_cost_cycle = (medical_cost_intervention_cycle - medical_cost_control_cycle) * coverage_scale
    incremental_total_cost_cycle = incremental_medical_cost_cycle + program_cost_monthly

    inc_cost_1y = float(np.dot(incremental_total_cost_cycle[:13], MONTHLY_DISCOUNT_1Y * CYCLE_WEIGHTS_1Y))
    daly_1y = float(np.dot(incremental_dalys_cycle[:13], MONTHLY_DISCOUNT_1Y * CYCLE_WEIGHTS_1Y))
    inc_cost_10y = float(np.dot(incremental_total_cost_cycle, MONTHLY_DISCOUNT_10Y * CYCLE_WEIGHTS_10Y))
    daly_10y = float(np.dot(incremental_dalys_cycle, MONTHLY_DISCOUNT_10Y * CYCLE_WEIGHTS_10Y))

    return {
        "scenario": scenario,
        "schedule": schedule,
        "Incremental_cost_1y_USD": inc_cost_1y,
        "DALYs_averted_1y": daly_1y,
        "ICER_1y_USD_per_DALY": safe_icer(inc_cost_1y, daly_1y),
        "Incremental_cost_10y_USD": inc_cost_10y,
        "DALYs_averted_10y": daly_10y,
        "ICER_10y_USD_per_DALY": safe_icer(inc_cost_10y, daly_10y),
    }


In [9]:
# ---------------------------------------------------------------------
# Exhibit builders
# ---------------------------------------------------------------------

def build_table_2() -> pd.DataFrame:
    rows = []
    for scenario in SCENARIOS:
        out = run_deterministic_scenario(scenario)
        rows.append({
            "Scenario_ID": scenario["Scenario_ID"],
            "Scenario": scenario["Scenario"],
            "Incremental_cost_1y_USD": out["Incremental_cost_1y_USD"],
            "DALYs_averted_1y": out["DALYs_averted_1y"],
            "ICER_1y_USD_per_DALY": out["ICER_1y_USD_per_DALY"],
            "Incremental_cost_10y_USD": out["Incremental_cost_10y_USD"],
            "DALYs_averted_10y": out["DALYs_averted_10y"],
            "ICER_10y_USD_per_DALY": out["ICER_10y_USD_per_DALY"],
        })
    return pd.DataFrame(rows)


def build_table_s1() -> pd.DataFrame:
    p = BASE_PARAMS
    p_uh_treated = p["p_UH"] + (p["p_UD"] - p["p_UD"] * (1.0 - p["act_eff"]))
    p_us_treated = p["p_US"]
    p_ud_treated = p["p_UD"] * (1.0 - p["act_eff"])

    rows = [
        ["Healthy", "Uncomplicated malaria", p["p_HU"], p["p_HU"], "Common across arms"],
        ["Healthy", "Death", p["p_HD"], p["p_HD"], "Common across arms"],
        ["Uncomplicated malaria", "Healthy", p["p_UH"], p_uh_treated, "Treated row used only when prompt ACT is received"],
        ["Uncomplicated malaria", "Severe malaria", p["p_US"], p_us_treated, "No treated adjustment in base model"],
        ["Uncomplicated malaria", "Death", p["p_UD"], p_ud_treated, "Reduced by ACT effectiveness"],
        ["Severe malaria", "Healthy", p["p_SH"], p["p_SH"], "Common across arms"],
        ["Severe malaria", "Death", p["p_SD"], p["p_SD"], "Common across arms"],
        ["Death", "Death", 1.0, 1.0, "Absorbing state"],
    ]
    return pd.DataFrame(rows, columns=[
        "From_state", "To_state", "Annual_probability_untreated", "Annual_probability_treated_if_applicable", "Note"
    ])


def build_table_s2() -> pd.DataFrame:
    sensitivity_values = [
        ("No additional ACT after additional testing", 0.10),
        ("Base-case pass-through", BASE_PARAMS["pass_through"]),
        ("Full pass-through from additional testing to prompt ACT", 1.00),
    ]
    scenario2 = next(s for s in SCENARIOS if s["Scenario_ID"] == 2)
    rows = []
    for label, val in sensitivity_values:
        out = run_deterministic_scenario(scenario2, pass_through=val)
        rows.append({
            "Scenario": label,
            "Pass_through": val,
            "Base_cost_scenario": scenario2["Scenario"],
            "Incremental_cost_10y_USD": out["Incremental_cost_10y_USD"],
            "DALYs_averted_10y": out["DALYs_averted_10y"],
            "ICER_10y_USD_per_DALY": out["ICER_10y_USD_per_DALY"],
        })
    return pd.DataFrame(rows)


In [10]:
# ---------------------------------------------------------------------
# Main routine
# ---------------------------------------------------------------------

def main() -> None:
    table2 = build_table_2()
    table_s1 = build_table_s1()
    table_s2 = build_table_s2()

    table2_path = "Table_2_deterministic_cost_effectiveness_results.csv"
    table_s1_path = "Table_S1_annual_transition_probabilities_and_treated_adjustment.csv"
    table_s2_path = "Table_S2_treatment_pass_through_sensitivity_analysis.csv"

    table2.to_csv(table2_path, index=False)
    table_s1.to_csv(table_s1_path, index=False)
    table_s2.to_csv(table_s2_path, index=False)

    print(f"Seed: {SEED}")
    print(f"Wrote: {table2_path}")
    print(f"Wrote: {table_s1_path}")
    print(f"Wrote: {table_s2_path}")
    print("\nTable 2 preview:")
    print(table2.to_string(index=False))


if __name__ == "__main__":
    main()


Seed: 12345
Wrote: Table_2_deterministic_cost_effectiveness_results.csv
Wrote: Table_S1_annual_transition_probabilities_and_treated_adjustment.csv
Wrote: Table_S2_treatment_pass_through_sensitivity_analysis.csv

Table 2 preview:
 Scenario_ID                                         Scenario  Incremental_cost_1y_USD  DALYs_averted_1y  ICER_1y_USD_per_DALY  Incremental_cost_10y_USD  DALYs_averted_10y  ICER_10y_USD_per_DALY
           1      Expensive pilot pricing + underused network            324315.200174        471.263473            688.182342              1.500175e+07       15212.710331             986.132751
           2 Transitional GAVI pricing + expected utilisation            324315.200174        471.263473            688.182342              1.266410e+07       32665.874625             387.686046
           3            At-scale pricing + efficient scale-up            324315.200174        471.263473            688.182342              7.201508e+06       36687.716258             19

In [11]:
def face_validation_exercise():
    print("--- Validation: Population & Incidence (Year 1) ---")
    print(f"Initial Under-5 Population: {initial_u5_population:,.0f}")

    # Extract service pathway metrics at baseline (utilization_scale = 1.0)
    pathway_y1 = service_pathway(
        utilization_scale=1.0,
        control_rdt_in_stock=BASE_PARAMS["control_rdt_in_stock"],
        delta_rdt_in_stock=BASE_PARAMS["delta_rdt_in_stock"],
        control_test_if_stock=BASE_PARAMS["control_test_if_stock"],
        delta_test_if_stock=BASE_PARAMS["delta_test_if_stock"],
        control_act_if_positive=BASE_PARAMS["control_act_if_positive"],
        pass_through=BASE_PARAMS["pass_through"]
    )

    # Prepare transition matrix for the control arm to track incidence
    transition_inputs = TransitionInputs(
        p_HD=BASE_PARAMS["p_HD"], p_HU=BASE_PARAMS["p_HU"], p_UH=BASE_PARAMS["p_UH"],
        p_US=BASE_PARAMS["p_US"], p_UD=BASE_PARAMS["p_UD"], p_SH=BASE_PARAMS["p_SH"],
        p_SD=BASE_PARAMS["p_SD"], act_eff=BASE_PARAMS["act_eff"]
    )

    P_control = build_transition_matrix(transition_inputs, pathway_y1["P_prompt_treated_control"])

    # Run the open cohort for just 12 months (Year 1)
    trace_control_y1 = run_open_cohort([P_control] * 12, births_per_month, initial_u5_population)

    # Calculate uncomplicated malaria episodes (Incidence = H_t * P(H -> U))
    p_h_to_u = P_control[H, U]
    year_1_episodes = sum(trace_control_y1[t, H] * p_h_to_u for t in range(12))

    episodes_per_1000 = (year_1_episodes / initial_u5_population) * 1000

    print(f"Predicted Uncomplicated Episodes: {year_1_episodes:,.2f}")
    print(f"Episodes per 1,000 Under-5s: {episodes_per_1000:.1f}\n")

    print("--- Validation: Service Pathway (Year 1) ---")
    p_tested_ctrl = pathway_y1["P_tested_control"] * 100
    p_tested_int = pathway_y1["P_tested_intervention"] * 100
    p_treated_ctrl = pathway_y1["P_prompt_treated_control"] * 100
    p_treated_int = pathway_y1["P_prompt_treated_intervention"] * 100

    print(f"Episodes tested (Control): {p_tested_ctrl:.1f}%")
    print(f"Episodes tested (Intervention): {p_tested_int:.1f}%")
    print(f"Prompt treatment (Control): {p_treated_ctrl:.1f}%")
    print(f"Prompt treatment (Intervention): {p_treated_int:.1f}%")

# Run the validation
face_validation_exercise()

--- Validation: Population & Incidence (Year 1) ---
Initial Under-5 Population: 87,480
Predicted Uncomplicated Episodes: 37,667.85
Episodes per 1,000 Under-5s: 430.6

--- Validation: Service Pathway (Year 1) ---
Episodes tested (Control): 48.6%
Episodes tested (Intervention): 75.8%
Prompt treatment (Control): 39.9%
Prompt treatment (Intervention): 62.2%
